# Hyperparameter Tuning

Runs a Hyperopt (TPE) search over LightGBM hyperparameters for the Home Credit
default-prediction model.

**Pipeline overview**
1. Load `model_dataset.parquet`
2. Train/validation split (fixed split reused by Notebook 3)
3. Define the search space & objective function
4. Run the Hyperopt search
5. Resolve & save the best hyperparameters
6. Save search history for diagnostics

**Output:**
- `data/HyperParameters/LightGBM_v1/lgbm_resolved_best.json` — best params found by the search
- `src/HyperParameters/LightGBM_v1/lgbm_trials_history.parquet` — full search history

> **Note on reproducibility (Bug 17 in the original):** the original notebook ran a
> live Hyperopt search and then silently overwrote the result with a hardcoded dict
> from a previous run — discarding the live search entirely without any indication.
> This notebook keeps **both** paths but makes the active one explicit via the
> `USE_PREVIOUS_BEST` flag in Section 1, and always prints which source is in effect.

## 1. Imports & Configuration

Set `USE_PREVIOUS_BEST = True` to skip the (slow) search and load a previously-saved
`lgbm_resolved_best.json` instead — useful for quickly re-running Notebook 3 without
re-tuning. Set it to `False` to run the full live search.

In [ ]:
import os
import gc; 
import json
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from hyperopt import fmin, tpe, hp, Trials, space_eval

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")
HPO_DIR  = os.path.join(DATA_ROOT, "HyperParameters", "LightGBM_v1")

os.makedirs(HPO_DIR, exist_ok=True)

PATH_MODEL_DATASET = os.path.join(DATA_ROOT, "ModelDataset", "model_dataset.parquet")
PATH_PARS_BEST = os.path.join(HPO_DIR, "lgbm_resolved_best.json")
PATH_TRIALS_HISTORY = os.path.join(HPO_DIR, "lgbm_trials_history.parquet")

# ── Reproducibility / split config — MUST match training exactly ────────
RANDOM_STATE = 1994
TEST_SIZE    = 0.20

# ── Active source for resolved_best ────────────────────────────────────────
# True  -> load a previously-saved lgbm_resolved_best.json (fast, skips the search)
# False -> run the live Hyperopt search below and save its result
USE_PREVIOUS_BEST = True
# pick a specific number of points from search space, try each one, and give me the best one you found
MAX_EVALS = 150

## 2. Load Data & Split

> **Fix (Bug 7 in the original):** the original used an 80/20 split here for HPO, but
> a *different* 85/15 split later for final training/evaluation in the training
> notebook — meaning the model selected via HPO was never actually validated on the
> exact data split used for final reporting. `RANDOM_STATE` and `TEST_SIZE` are now
> defined once above and must be reused identically in Notebook 3.

In [2]:
model_dataset = pd.read_parquet(PATH_MODEL_DATASET)

# Re-derive cat_cols and re-apply 'category' dtype — parquet round-trips pandas
# categories correctly, but we re-assert here defensively in case the file was
# regenerated by a different process.
cat_cols = model_dataset.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    model_dataset[col] = model_dataset[col].astype('category')

feature_cols = [c for c in model_dataset.columns if c not in ('SK_ID_CURR', 'TARGET')]
X = model_dataset[feature_cols]
y = model_dataset['TARGET']

print(f"X shape: {X.shape}")
print(f"Categorical columns: {len(cat_cols)}")

X shape: (307511, 645)
Categorical columns: 14


Convert `float64` type columns to `float32`

In [5]:
# ADD THIS TO SHRINK THE DATAFRAME IN MEMORY:
float_cols = X.select_dtypes(include=['float64']).columns
X[float_cols] = X[float_cols].astype('float32')

C:\Users\domjo\AppData\Local\Temp\ipykernel_31124\3389940897.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[float_cols] = X[float_cols].astype('float32')


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")

In [ ]:
del X, y, model_dataset
gc.collect()

## 3. Define Search Space

Boosting type is fixed to `'goss'` (Gradient-based One-Side Sampling), which does
**not** use row subsampling — `subsample` therefore has zero effect under GOSS.

> **Fix (Bug 18 in the original):** the original included `subsample` in the search
> space anyway, wasting search budget exploring a parameter with no effect, then
> deleted it from the result after the fact. It is excluded from the space entirely here.

In [ ]:
BOOSTING_TYPE = 'goss'  # GOSS ignores subsample — excluded from the space below

space = {
    # Core tree structure
    'num_leaves': hp.quniform('num_leaves', 20, 150, 1),
    'max_depth':  hp.quniform('max_depth', 4, 12, 1),

    # Learning rate & boosting iterations
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),
    'n_estimators':  hp.quniform('n_estimators', 100, 1000, 10),

    # Feature sampling (GOSS handles row sampling internally — only column sampling tuned here)
    'colsample_bytree': hp.uniform('colsample_bytree', 0.1, 1.0),

    # Regularization
    'reg_alpha':  hp.loguniform('reg_alpha', np.log(1e-3), np.log(100.0)),
    'reg_lambda': hp.loguniform('reg_lambda', np.log(1e-3), np.log(100.0)),

    # Controls adding more splits/leaves to one tree. If the calculated Gain is less than your min_split_gain, LightGBM says, 
    # "This split isn't worth the effort," and refuses to split that leaf.
    'min_split_gain ': hp.loguniform('min_split_gain', np.log(1e-3), np.log(1.0)),

    # Overfitting control for wide datasets
    'min_child_samples': hp.quniform('min_child_samples', 20, 500, 10),
}

INT_PARAMS = ['num_leaves', 'max_depth', 'n_estimators', 'min_child_samples']

## 4. Objective Function

In [ ]:
def objective(params):
    """
    Hyperopt objective: 5-fold CV mean AUC for one LightGBM hyperparameter set.

    Hyperopt minimizes the return value, so we negate AUC (higher AUC = lower loss).
    Manual K-fold (rather than cross_val_score) is used so each fold can use
    LightGBM's native early stopping.
    """
    current_params = params.copy()
    for p in INT_PARAMS:
        current_params[p] = int(current_params[p])

    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
    auc_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(
            objective='binary',
            boosting_type=BOOSTING_TYPE,
            max_bin=63,
            device='cpu',
            seed=42,
            n_jobs=14,#Leave cores available for Airflow
            verbose=-1,
            **current_params,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping( #controls building of enough trees
                stopping_rounds=15, #if 15 new trees are built without improvement of AUC by min_delta then stop further ensembles
                verbose=False,
                first_metric_only=True, # Recommended when using min_delta
                min_delta=0.0001 # minimum metric improvement needed to add new trees
                )
                ],
        )
        
        preds = model.predict_proba(X_va)[:, 1]
        auc_scores.append(roc_auc_score(y_va, preds))
        # ADD THIS: Delete the fold data and force garbage collection
        del X_tr, y_tr, X_va, y_va, model
        gc.collect()


    mean_auc = np.mean(auc_scores)
    return {'loss': -mean_auc, 'status': 'ok'}

## 5. Run / Load Search

Controlled by `USE_PREVIOUS_BEST` in Section 1. Either branch produces a single,
clearly-labelled `resolved_best` dict and prints which source produced it — there is
no silent overwrite as in the original notebook.

In [ ]:
if USE_PREVIOUS_BEST:
    print(f"USE_PREVIOUS_BEST=True → loading saved params from {PATH_PARS_BEST}")
    with open(PATH_PARS_BEST) as f:
        resolved_best = json.load(f)
    resolved_best_source = 'loaded_from_json'
    trials = None

else:
    print(f"USE_PREVIOUS_BEST=False → running live Hyperopt search ({MAX_EVALS} evals)")
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=MAX_EVALS,
        trials=trials,
    )
    print("\n--- Hyperopt Search Complete ---")

    resolved_best = space_eval(space, best)
    for p in INT_PARAMS:
        resolved_best[p] = int(resolved_best[p])
    resolved_best_source = 'live_search'

print(f"\nActive resolved_best source: {resolved_best_source}")
print(json.dumps(resolved_best, indent=4))

## 6. Save Results

Always writes the *active* `resolved_best` (whichever branch ran above) to disk,
along with the full trial history when a live search was run.

> **Fix (Bug 16 in the original):** `trials_df` / `trials_history` were computed but
> never saved — discarding useful diagnostic data about the search trajectory. Now
> saved to `lgbm_trials_history.parquet` whenever a live search runs.

In [ ]:
if not USE_PREVIOUS_BEST:
    with open(PATH_PARS_BEST, 'w') as f:
        json.dump(resolved_best, f, indent=4)
        print(f"Saved resolved_best ({resolved_best_source}) → {PATH_PARS_BEST}")

if trials is not None:
    trials_df = pd.DataFrame(trials.results)
    trials_df['auc'] = -trials_df['loss']
    trials_df['trial_number'] = range(len(trials_df))

    params_df = pd.DataFrame([t['misc']['vals'] for t in trials.trials])
    params_df = params_df.applymap(lambda x: x[0] if isinstance(x, list) and x else None)

    trials_history = pd.concat([trials_df, params_df], axis=1)
    trials_history.to_parquet(PATH_TRIALS_HISTORY, index=False)
    print(f"Saved trial history ({len(trials_history)} trials) → {PATH_TRIALS_HISTORY}")

    print(f"\nBest CV AUC found: {trials_history['auc'].max():.4f}")
else:
    print("No live search ran — trial history not available this run.")

## 7. Cleanup

In [ ]:
del X_train, X_val, y_train, y_val
gc.collect()
print("All DataFrames released.")